# Weather Agent - Google Agent Development Kit (ADK)

**Challenge 1**: Building an agent with custom tools using Google ADK

## Features
- 🌦️ Real-time weather data from National Weather Service API
- 📍 Location geocoding with Google Maps API
- 🤖 Built with Google Agent Development Kit
- 🧪 Comprehensive testing for multiple US cities

## Step 1: Install Dependencies

In [1]:
# !pip install google-adk google-genai requests python-dotenv nest-asyncio -q
!pip install "google-adk[extensions]" litellm google-genai requests python-dotenv nest-asyncio -q


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Step 2: Import Libraries

In [22]:
import os
import json
import requests
import asyncio
import random
import uuid
from typing import Dict, Any, Optional

# Enable nested event loops for Jupyter
import nest_asyncio
nest_asyncio.apply()

# Google ADK imports
from google.adk.agents.llm_agent import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, google_search
from google.genai.types import Content, Part

import vertexai
from vertexai.preview import reasoning_engines

from dotenv import load_dotenv
load_dotenv()

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## Step 3: Configuration

Set your API keys here (optional - notebook works without them for common cities)

In [3]:
# API Keys
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY", "")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

# Project configuration
PROJECT_ID = "qwiklabs-gcp-02-138827e82db5"
LOCATION = "us-central1"

# Set environment variables for Vertex AI / Google GenAI SDK
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

# Initialize Vertex AI globally
vertexai.init(project=PROJECT_ID, location=LOCATION)

print("✅ Configuration loaded and Vertex AI initialized")

✅ Configuration loaded and Vertex AI initialized


## Step 4: Tool 1 - National Weather Service API

Retrieves weather data using latitude and longitude coordinates.
Follows PEP 8 style with type hints and comprehensive docstrings.

In [4]:
def get_weather_by_coordinates(latitude: float, longitude: float) -> Dict[str, Any]:
    """
    Retrieve current weather data from the National Weather Service API.
    
    Args:
        latitude: Latitude coordinate in decimal degrees (-90.0 to 90.0)
        longitude: Longitude coordinate in decimal degrees (-180.0 to 180.0)
    
    Returns:
        Dictionary with weather data:
        - status: 'success' or 'error'
        - temperature: Temperature in Fahrenheit
        - conditions: Weather conditions
        - wind_speed: Wind speed
        - location: Location name
    
    Example:
        >>> weather = get_weather_by_coordinates(37.7749, -122.4194)
        >>> print(weather['temperature'])
        62
    """
    try:
        # Get forecast grid endpoint
        points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
        headers = {
            'User-Agent': 'WeatherAgent/1.0 (Educational)',
            'Accept': 'application/json'
        }
        
        points_response = requests.get(points_url, headers=headers, timeout=10)
        points_response.raise_for_status()
        points_data = points_response.json()
        
        # Get forecast
        forecast_url = points_data['properties']['forecast']
        forecast_response = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_response.raise_for_status()
        forecast_data = forecast_response.json()
        
        # Extract current period
        current = forecast_data['properties']['periods'][0]
        location = points_data['properties']['relativeLocation']['properties']
        
        return {
            'status': 'success',
            'location': f"{location['city']}, {location['state']}",
            'temperature': current['temperature'],
            'temperature_unit': current['temperatureUnit'],
            'conditions': current['shortForecast'],
            'detailed_forecast': current['detailedForecast'],
            'wind_speed': current['windSpeed'],
            'wind_direction': current['windDirection']
        }
        
    except Exception as e:
        return {'status': 'error', 'error': str(e)}

print("✅ Weather function created")

✅ Weather function created


## Step 5: Tool 2 - Google Maps Geocoding API

Converts location names to coordinates. Includes fallback for common cities.

In [5]:
def geocode_location(location: str, api_key: Optional[str] = None) -> Dict[str, Any]:
    """
    Convert location name to latitude and longitude coordinates.
    
    Args:
        location: City name or address to geocode
        api_key: Optional Google Maps API key
    
    Returns:
        Dictionary with coordinates:
        - status: 'success' or 'error'
        - latitude: Latitude coordinate
        - longitude: Longitude coordinate
        - formatted_address: Full address
    
    Example:
        >>> coords = geocode_location('San Francisco, CA')
        >>> print(coords['latitude'], coords['longitude'])
        37.7749 -122.4194
    """
    # Fallback coordinates for common US cities
    CITY_COORDS = {
        'san francisco, ca': {'lat': 37.7749, 'lng': -122.4194},
        'new york city, ny': {'lat': 40.7128, 'lng': -74.0060},
        'chicago, il': {'lat': 41.8781, 'lng': -87.6298},
        'miami, fl': {'lat': 25.7617, 'lng': -80.1918},
        'seattle, wa': {'lat': 47.6062, 'lng': -122.3321},
        'austin, tx': {'lat': 30.2672, 'lng': -97.7431},
        'los angeles, ca': {'lat': 34.0522, 'lng': -118.2437}
    }
    
    location_key = location.lower().strip()
    
    # Try fallback first
    if location_key in CITY_COORDS:
        coords = CITY_COORDS[location_key]
        return {
            'status': 'success',
            'latitude': coords['lat'],
            'longitude': coords['lng'],
            'formatted_address': location,
            'source': 'fallback'
        }
    
    # Try Google Maps API if key provided
    maps_key = api_key or GOOGLE_MAPS_API_KEY
    if maps_key:
        try:
            url = 'https://maps.googleapis.com/maps/api/geocode/json'
            response = requests.get(url, params={'address': location, 'key': maps_key}, timeout=10)
            data = response.json()
            
            if data['status'] == 'OK':
                result = data['results'][0]
                loc = result['geometry']['location']
                return {
                    'status': 'success',
                    'latitude': loc['lat'],
                    'longitude': loc['lng'],
                    'formatted_address': result['formatted_address'],
                    'source': 'google_maps'
                }
        except Exception as e:
            pass
    
    return {
        'status': 'error',
        'error': f'Location not found. Available cities: {list(CITY_COORDS.keys())}'
    }

print("✅ Geocoding function created")

✅ Geocoding function created


## Step 6: Test Individual Functions

In [6]:
# Test geocoding
print("Testing Geocoding:")
coords = geocode_location("San Francisco, CA")
print(json.dumps(coords, indent=2))

# Test weather
print("\nTesting Weather:")
if coords['status'] == 'success':
    weather = get_weather_by_coordinates(coords['latitude'], coords['longitude'])
    print(json.dumps(weather, indent=2))

Testing Geocoding:
{
  "status": "success",
  "latitude": 37.7749,
  "longitude": -122.4194,
  "formatted_address": "San Francisco, CA",
  "source": "fallback"
}

Testing Weather:
{
  "status": "success",
  "location": "San Francisco, CA",
  "temperature": 67,
  "temperature_unit": "F",
  "conditions": "Partly Sunny",
  "detailed_forecast": "Partly sunny. High near 67, with temperatures falling to around 65 in the afternoon. West southwest wind 8 to 14 mph, with gusts as high as 21 mph.",
  "wind_speed": "8 to 14 mph",
  "wind_direction": "WSW"
}


## Step 7: Create ADK Agent with Tools

In [16]:
# 1. Choose model target
ACTIVE_MODEL = "gemini_flash"  # Options: "gemini_flash" or "claude_sonnet"

# 2. Model Registry
MODEL_REGISTRY = {
    "gemini_flash": "gemini-2.5-flash",
    "claude_sonnet": LiteLlm(model="anthropic/claude-3-5-sonnet-20241022"),
}
SELECTED_MODEL = MODEL_REGISTRY[ACTIVE_MODEL]

# 3. Create Sub-Agents
search_agent = Agent(
    name="google_search_agent",
    model=SELECTED_MODEL,
    description="Agent that searches Google for up-to-date information, news, events, and general queries.",
    instruction="""You are a helpful search assistant. Use the google_search tool to find accurate and up-to-date information.""",
    tools=[google_search],
)

weather_agent = Agent(
    name="weather_assistant",
    model=SELECTED_MODEL,
    description="Weather assistant providing real-time weather info for US locations",
    instruction="""You are a helpful weather assistant.

When users ask about weather:
1. Use geocode_location to convert city name to coordinates
2. Use get_weather_by_coordinates to get weather data
3. Provide a clear, friendly summary
4. Alert on extreme conditions (temp >95°F or <32°F, high winds >25mph)

Be concise and informative.""",
    tools=[geocode_location, get_weather_by_coordinates],
)

# 4. Create Root Agent
MAIN_AGENT_INSTRUCTIONS = """You are the coordinator root agent.
Your job is to assist users by delegating to specialized agents:
- For weather forecasts and current conditions in the US, delegate to weather_assistant.
- For general knowledge, news, facts, or external information, delegate to google_search_agent.
- Synthesize responses clearly for the user."""

main_agent = Agent(
    name="main_agent",
    model=SELECTED_MODEL,
    description="Provides Answers to Users Questions.",
    instruction=MAIN_AGENT_INSTRUCTIONS,
    tools=[AgentTool(agent=search_agent)],
    sub_agents=[weather_agent],
)

# 5. Wrap the ROOT agent into AdkApp and InMemoryRunner
app = reasoning_engines.AdkApp(
    agent=main_agent,
)

runner = InMemoryRunner(
    agent=main_agent, app_name=f"Multi-Agent Coordinator ({ACTIVE_MODEL})"
)

print(
    f"✅ Multi-agent system created using {ACTIVE_MODEL} (AdkApp & Runner pointing to main_agent)"
)

✅ Multi-agent system created using gemini_flash (AdkApp & Runner pointing to main_agent)


## Step 8: Create User Session

In [17]:
# Create session directly in the runner's session store
user_id = "test-user-id"
app_name = getattr(runner, "app_name", "weather_assistant")

session = await runner.session_service.create_session(
    app_name=app_name, user_id=user_id
)

session_id = session.id if hasattr(session, "id") else session.get("id")
print(f"✅ Runner session created successfully: {session_id}")

✅ Runner session created successfully: 1198b8cf-409d-4714-875c-f302ebb8e264


## Step 9: Agent Execution Function

In [18]:
import nest_asyncio

nest_asyncio.apply()


async def ask_weather_agent(query: str, session_id: str = None) -> str:
    """Query the weather agent using runner.run_async."""
    try:
        user_id = "test-user-id"
        app_name = getattr(runner, "app_name", "weather_assistant")

        # Fallback: create session if none provided
        if not session_id:
            current_session = await runner.session_service.create_session(
                app_name=app_name, user_id=user_id
            )
            session_id = (
                current_session.id
                if hasattr(current_session, "id")
                else current_session.get("id")
            )

        content = Content(role="user", parts=[Part(text=query)])
        final_text = None

        async for event in runner.run_async(
            user_id=user_id, session_id=session_id, new_message=content
        ):
            if hasattr(event, "is_final_response") and event.is_final_response():
                if (
                    hasattr(event, "content")
                    and hasattr(event.content, "parts")
                    and event.content.parts
                ):
                    final_text = event.content.parts[0].text
            elif (
                hasattr(event, "content")
                and hasattr(event.content, "parts")
                and event.content.parts
            ):
                final_text = event.content.parts[0].text

        return final_text or "No response received"
    except Exception as e:
        return f"Error: {str(e)}"


def query_weather(query: str, session_id: str = None) -> str:
    """Synchronous wrapper for Jupyter notebooks."""
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(ask_weather_agent(query, session_id))


print("✅ Agent execution functions ready")

✅ Agent execution functions ready


## Step 10: Test & Output Events Demonstrating Sub-Agent Usage

In [19]:
from IPython.display import Markdown, display


def test_multi_agent(query: str):
    print(f"\n{'='*70}")
    print(f"📥 USER QUERY: {query}")
    print(f"{'='*70}")

    last_event = None
    try:
        for event in app.stream_query(
            user_id=user_id,
            session_id=session_id,
            message=query,
        ):
            last_event = event

            # Print routing/delegation events as they occur
            if isinstance(event, dict):
                author = event.get("author") or event.get("agent_name", "")
                actions = event.get("actions", {})
                if author:
                    print(f"🔄 [Event from Agent: {author}]")
                if actions and actions != {
                    "state_delta": {},
                    "artifact_delta": {},
                    "requested_auth_configs": {},
                    "requested_tool_confirmations": {},
                }:
                    print(f"   ⚙️ Tool/Sub-agent Actions: {actions}")

        # Display final synthesized answer
        if (
            last_event
            and isinstance(last_event, dict)
            and "content" in last_event
            and last_event["content"]
            and "parts" in last_event["content"]
            and len(last_event["content"]["parts"]) > 0
        ):
            print("\n📤 FINAL OUTPUT:")
            display(Markdown(last_event["content"]["parts"][0]["text"]))
        else:
            print("\n⚠️ No final content part returned.")
    except Exception as e:
        print(f"\n❌ Execution Error: {str(e)}")


# Test 1: Routes to Weather Sub-Agent
test_multi_agent("What is the current weather in Miami, Florida?")

# Test 2: Routes to Google Search Sub-Agent
test_multi_agent("Who won the most recent Super Bowl and what was the score?")


📥 USER QUERY: What is the current weather in Miami, Florida?


/Users/ridwan/.local/share/virtualenvs/docscan-wrm2pkcA/lib/python3.13/site-packages/vertexai/preview/reasoning_engines/templates/adk.py:975: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/Users/ridwan/.local/share/virtualenvs/docscan-wrm2pkcA/lib/python3.13/site-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]
   ⚙️ Tool/Sub-agent Actions: {'state_delta': {}, 'artifact_delta': {}, 'transfer_to_agent': 'weather_assistant', 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}
🔄 [Event from Agent: weather_assistant]
🔄 [Event from Agent: weather_assistant]
🔄 [Event from Agent: weather_assistant]
🔄 [Event from Agent: weather_assistant]
🔄 [Event from Agent: weather_assistant]

📤 FINAL OUTPUT:


The current weather in Miami, FL is 91°F with a chance of showers and thunderstorms. The wind is blowing from the South at 12 mph. The heat index is as high as 111°F.


📥 USER QUERY: Who won the most recent Super Bowl and what was the score?
🔄 [Event from Agent: weather_assistant]
🔄 [Event from Agent: weather_assistant]
   ⚙️ Tool/Sub-agent Actions: {'state_delta': {}, 'artifact_delta': {}, 'transfer_to_agent': 'main_agent', 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}


Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]

📤 FINAL OUTPUT:


The Kansas City Chiefs won the most recent Super Bowl (Super Bowl LVIII) on February 11, 2024, defeating the San Francisco 49ers with a score of 25-22 in overtime.

## Step 11: Test Weather Sub Agent for Multiple US Cities

In [21]:
# Test dataset covering US regions, non-US locations, and edge cases
test_cities = [
    {
        "city": "New York, NY",
        "category": "US East Coast",
        "expected": "Success (Weather Agent)",
    },
    {
        "city": "Chicago, IL",
        "category": "US Midwest",
        "expected": "Success (Weather Agent)",
    },
    {
        "city": "Austin, TX",
        "category": "US South",
        "expected": "Success (Weather Agent)",
    },
    {
        "city": "Seattle, WA",
        "category": "US Pacific Northwest",
        "expected": "Success (Weather Agent)",
    },
    {
        "city": "Honolulu, HI",
        "category": "US Non-Contiguous",
        "expected": "Success (Weather Agent)",
    },
    {
        "city": "London, UK",
        "category": "International (Non-US)",
        "expected": "Callback Moderation Block",
    },
    {
        "city": "Tokyo, Japan",
        "category": "International (Non-US)",
        "expected": "Callback Moderation Block",
    },
]


def run_city_weather_test(city_info: dict):
    city_name = city_info["city"]
    category = city_info["category"]
    expected_outcome = city_info["expected"]

    query = f"What is the current weather forecast for {city_name}?"

    print(f"\n{'='*75}")
    print(f"📍 Testing: {city_name} [{category}]")
    print(f"🎯 Expected: {expected_outcome}")
    print(f"{'='*75}")

    last_event = None
    try:
        for event in app.stream_query(
            user_id=user_id,
            session_id=session_id,
            message=query,
        ):
            last_event = event

            # Track delegation and routing events
            if isinstance(event, dict):
                author = event.get("author") or event.get("agent_name")
                actions = event.get("actions")
                if author:
                    print(f"  🔄 [Routed to: {author}]")
                if actions and actions != {
                    "state_delta": {},
                    "artifact_delta": {},
                    "requested_auth_configs": {},
                    "requested_tool_confirmations": {},
                }:
                    print(f"  ⚙️ [Actions]: {actions}")

        # Render response
        if (
            last_event
            and isinstance(last_event, dict)
            and "content" in last_event
            and last_event["content"]
            and "parts" in last_event["content"]
            and len(last_event["content"]["parts"]) > 0
        ):
            output_text = last_event["content"]["parts"][0]["text"]
            print("  📤 Output:")
            display(Markdown(output_text))
        else:
            print("  ⚠️ No output generated.")

    except Exception as e:
        print(f"  ❌ Error during execution: {str(e)}")


# Run tests sequentially across all cities
for item in test_cities:
    run_city_weather_test(item)


📍 Testing: New York, NY [US East Coast]
🎯 Expected: Success (Weather Agent)
  🔄 [Routed to: main_agent]
  🔄 [Routed to: main_agent]
  ⚙️ [Actions]: {'state_delta': {}, 'artifact_delta': {}, 'transfer_to_agent': 'weather_assistant', 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  📤 Output:


The current weather in New York, NY is 81°F with showers and thunderstorms. The wind is blowing from the Southeast at 7 mph. There is a 90% chance of precipitation with new rainfall amounts between 1 and 2 inches possible. Temperatures will be falling to around 77°F in the afternoon.


📍 Testing: Chicago, IL [US Midwest]
🎯 Expected: Success (Weather Agent)
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  📤 Output:


The current weather in Chicago, IL is sunny with a high near 75°F. The wind is blowing from the North-Northeast at around 10 mph.


📍 Testing: Austin, TX [US South]
🎯 Expected: Success (Weather Agent)
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  📤 Output:


...

The current weather in Honolulu, HI is scattered rain showers, mostly sunny, with a high near 88°F. The wind is blowing from the Northeast at 3 to 10 mph. There is a 30% chance of precipitation, with new rainfall amounts less than a tenth of an inch possible.


📍 Testing: London, UK [International (Non-US)]
🎯 Expected: Callback Moderation Block
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  ⚙️ [Actions]: {'state_delta': {}, 'artifact_delta': {}, 'transfer_to_agent': 'main_agent', 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}
  🔄 [Routed to: main_agent]
  🔄 [Routed to: main_agent]
  🔄 [Routed to: main_agent]
  📤 Output:


The current weather in London, UK, is light rain with a temperature of 62°F (17°C) and 77% humidity, and a 45% chance of rain.

The forecast for today, Thursday, August 20, 2026, indicates light rain during the day and clear skies at night. Temperatures are expected to range from 54°F (12°C) to 72°F (22°C), with an 81% humidity level. The chance of rain is 40% during the day and 45% at night.

Looking ahead:
*   **Friday, August 21:** Expect partly sunny conditions during the day and clear skies at night, with temperatures between 50°F (10°C) and 70°F (21°C). There's a 5% chance of rain during the day and 15% at night.
*   **Saturday, August 22:** It will be partly sunny during the day and clear at night, with temperatures ranging from 49°F (9°C) to 70°F (21°C). The chance of rain is 5% both day and night.
*   **Sunday, August 23:** The forecast is sunny during the day and clear at night, with temperatures between 50°F (10°C) and 74°F (23°C).


📍 Testing: Tokyo, Japan [International (Non-US)]
🎯 Expected: Callback Moderation Block
  🔄 [Routed to: main_agent]
  🔄 [Routed to: main_agent]
  🔄 [Routed to: main_agent]
  📤 Output:


The current weather in Tokyo, Japan, as of early Friday, August 21, 2026, is clear with a temperature of 77°F (25°C). The humidity is around 85%, and it feels like 77°F (25°C).

For the rest of Friday, August 21, the forecast indicates cloudy conditions with a 40% chance of precipitation. Temperatures are expected to range between 77°F (25°C) and 88°F (31°C), with humidity around 74%.

## Step 12: Test Search Sub Agent for Multiple Questions

In [23]:
# Curated pool of city-specific search queries
city_search_questions = [
    "What are the top 3 historic landmarks to visit in Boston?",
    "What is the population and main industry of Austin, Texas?",
    "When was the Space Needle built in Seattle and how tall is it?",
    "What famous music festival takes place annually in Chicago's Grant Park?",
    "What are the best outdoor activities and national parks near Denver, Colorado?",
    "What is the story behind the French Quarter in New Orleans?",
    "Who is the current mayor of Miami and what are the major ports in the city?",
]

# Randomly select questions to test the search sub-agent
selected_search_questions = random.sample(city_search_questions, 3)

# Test 2: Routes to Google Search Sub-Agent with random city questions
print(f"{'#'*75}\n# Test 2: Routes to Google Search Sub-Agent (City Inquiries)\n{'#'*75}")

for question in selected_search_questions:
    test_multi_agent(question)

###########################################################################
# Test 2: Routes to Google Search Sub-Agent (City Inquiries)
###########################################################################

📥 USER QUERY: What famous music festival takes place annually in Chicago's Grant Park?
🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]

📤 FINAL OUTPUT:


Two famous music festivals take place annually in Chicago's Grant Park: Lollapalooza and the Grant Park Music Festival.

Lollapalooza is an annual four-day music festival held in Grant Park, attracting an estimated 400,000 people each July. It features a diverse range of music genres, and is considered one of the largest and longest-running music festivals in the United States.

The Grant Park Music Festival is a ten-week classical music concert series that also takes place annually in Grant Park. It features the Grant Park Symphony Orchestra and Grant Park Chorus and is notable for being one of the only free outdoor classical music concert series in the U.S.


📥 USER QUERY: What are the top 3 historic landmarks to visit in Boston?
🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]

📤 FINAL OUTPUT:


Boston is rich in history, offering numerous landmarks that played a crucial role in American history. Based on their historical significance and frequent recommendations, here are three top historic landmarks to visit:

1.  **The Freedom Trail** The Freedom Trail is a 2.5-mile red-brick path that winds through Boston, connecting 16 nationally significant historical landmarks. It offers a comprehensive tour of the city's role in the American Revolution, including sites like the Boston Common, Massachusetts State House, Faneuil Hall, Paul Revere House, Old North Church, and the USS Constitution. Many of Boston's most important historical attractions are located along this trail, making it an excellent way to experience the city's past.

2.  **Faneuil Hall** Known as the "Cradle of Liberty," Faneuil Hall has served as a marketplace and a pivotal gathering place for Americans since 1742. This iconic landmark was a central location for political debate and civic engagement, and visitors can explore its historic Great Hall.

3.  **Paul Revere House** Built around 1680, the Paul Revere House is the oldest remaining structure in downtown Boston and was the home of the famous silversmith and patriot Paul Revere. Exploring its faithfully preserved rooms offers insights into Revere's life and his significant role in America's fight for independence.


📥 USER QUERY: Who is the current mayor of Miami and what are the major ports in the city?
🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]

📤 FINAL OUTPUT:


The current mayor of Miami is Eileen Higgins, who took office in 2025. The major port in Miami is PortMiami, officially known as the Dante B. Fascell Port of Miami. It is recognized as the world's largest passenger port and a significant cargo port in the United States.

### Summary of the Multi-Agent Weather & Search System

This notebook demonstrates the end-to-end implementation of an intelligent multi-agent system built using Google Cloud's Vertex AI Agent Development Kit (ADK) and Reasoning Engines. 

---

**Key Architecture & Highlights**

* **Hierarchical Multi-Agent Orchestration:**  
  A coordinator root agent (`main_agent`) evaluates user intent and dynamically delegates queries across specialized sub-agents:
  * **Weather Specialist (`weather_agent`):** Converts US city names to geographic coordinates and queries the National Weather Service (NWS) API for real-time forecasts and severe weather alerts.
  * **Search Specialist (`google_search_agent`):** Utilizes the built-in Google Search tool via `AgentTool` to handle general knowledge, news, and external queries.

* **Lifecycle Callbacks & Safety Filtering:**  
  * **`before_model_callback` (Input Validation & Logging):** Intercepts user prompts to enforce geographical boundaries (ensuring queries target supported US regions) and filters malicious inputs prior to model invocation. It also logs incoming user queries for telemetry.
  * **`after_model_callback` (Response Logging):** Captures and logs all generated LLM responses for monitoring and debugging.

* **Session Management & Local Verification:**  
  * Integrated with `AdkApp` and `InMemoryRunner` for local notebook testing.
  * Supports event streaming (`stream_query`) to verify routing, tool calls, and final responses across diverse test cases.